# 🌿 AGRILENS — MobileNetV4 24-Class Retraining Pipeline
### Single Source of Truth: `MobileV4/datasets` (15 Epochs on Google Colab T4 GPU)

This notebook trains a **MobileNetV4-Conv-Small** vision backbone strictly on the **4 species folders** (`24 classes`, `23,893 images`) inside `MobileV4/datasets`:
1. **`apple dataset`** (`Apple_Rust`, `Apple_Scab`, `Apple_healthy` — 1,730 images)
2. **`mango dataset`** (`Mango_Anthracnose`, `Mango_Bacterial_Canker`, `Mango_Cutting_Weevil`, `Mango_Die_Back`, `Mango_Gall_Midge`, `Mango_Healthy`, `Mango_Powdery_Mildew`, `Mango_Sooty_Mould` — 4,000 images)
3. **`Potato Dataset`** (`Potato_Early_blight`, `Potato_Late_blight`, `Potato_healthy` — 2,152 images)
4. **`Tomato dataset`** (`Tomato_Bacterial_spot`, `Tomato_Early_blight`, `Tomato_Late_blight`, `Tomato_Leaf_Mold`, `Tomato_Septoria_leaf_spot`, `Tomato_Spider_mites_Two_spotted_spider_mite`, `Tomato_Target_Spot`, `Tomato_Tomato_YellowLeaf_Curl_Virus`, `Tomato_Tomato_mosaic_virus`, `Tomato_healthy` — 16,011 images)

**Hardware Requirement:** Free T4 GPU (`Runtime -> Change runtime type -> T4 GPU`).
**Training Time:** ~12–15 minutes for 15 epochs on T4 GPU.

In [ ]:
# 1. Install PyTorch Image Models (timm), ONNX & ONNXScript
!pip install -q timm onnx onnxruntime onnxscript pillow


In [ ]:
# 2. Verify GPU Acceleration
import torch
print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"✅ Active GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ WARNING: No GPU detected! Please go to: Runtime -> Change runtime type -> Select T4 GPU")


In [ ]:
# 3. Locate Dataset from Offline Computer Drive OR Upload to Colab
import os, glob
from pathlib import Path

REQUIRED_SPECIES = ['apple dataset', 'mango dataset', 'Potato Dataset', 'Tomato dataset']

# 1) Check Offline Local Computer Paths first (works when running locally or via Colab Local Runtime)
local_candidates = [
    r'G:\My Drive\EXPISCOR 2026\MobileV4\datasets',
    r'D:\Shreyance\Antigravity IDE\AGRILENS\datasets',
    str(Path.cwd() / 'datasets'),
    '/content/dataset',
]

DRIVE_DATA_DIR = None
TAR_BUNDLE_PATH = None

for c in local_candidates:
    if os.path.isdir(c) and all(os.path.isdir(os.path.join(c, sp)) for sp in REQUIRED_SPECIES):
        DRIVE_DATA_DIR = c
        break

# 2) If running in Cloud Google Colab, check /content or Google Drive, or prompt to pick from computer
if not DRIVE_DATA_DIR:
    if os.path.exists('/content/datasets_24classes_colab.tar'):
        TAR_BUNDLE_PATH = Path('/content/datasets_24classes_colab.tar')
    else:
        try:
            from google.colab import drive, files
            try:
                drive.mount('/content/drive', force_remount=False)
            except Exception:
                pass
            colab_candidates = [
                '/content/drive/MyDrive/EXPISCOR 2026/MobileV4/datasets',
                '/content/drive/My Drive/EXPISCOR 2026/MobileV4/datasets',
                '/content/drive/Shareddrives/EXPISCOR 2026/MobileV4/datasets',
            ]
            for c in colab_candidates:
                if os.path.isdir(c) and all(os.path.isdir(os.path.join(c, sp)) for sp in REQUIRED_SPECIES):
                    DRIVE_DATA_DIR = c
                    break
            if not DRIVE_DATA_DIR:
                matches = glob.glob('/content/drive/**/datasets_24classes_colab.tar', recursive=True)
                if matches:
                    TAR_BUNDLE_PATH = Path(matches[0])
            if not DRIVE_DATA_DIR and not TAR_BUNDLE_PATH:
                print("📂 Google Drive account does not have EXPISCOR 2026.")
                print("👉 Please select 'G:\\My Drive\\EXPISCOR 2026\\MobileV4\\datasets_24classes_colab.tar' from your computer below:")
                uploaded = files.upload()
                for fname in uploaded.keys():
                    TAR_BUNDLE_PATH = Path('/content') / fname
                    break
        except ImportError:
            pass

if DRIVE_DATA_DIR:
    print("✅ Located Dataset Directory:", DRIVE_DATA_DIR)
elif TAR_BUNDLE_PATH and TAR_BUNDLE_PATH.exists():
    print("✅ Located Offline Dataset Bundle:", TAR_BUNDLE_PATH)
else:
    raise FileNotFoundError("Could not locate dataset. In Colab, upload 'G:\\My Drive\\EXPISCOR 2026\\MobileV4\\datasets_24classes_colab.tar' to /content/.")


In [ ]:
# 4. Stage All 24 Classes from the 4 Species Folders into Local Colab NVMe (/content/dataset)
import shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import tempfile
WORK_ROOT = Path('/content') if os.path.exists('/content') else (Path.home() / '.agrilens_colab_work')
WORK_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_DATA = WORK_ROOT / 'dataset'
if LOCAL_DATA.exists():
    shutil.rmtree(LOCAL_DATA)
LOCAL_DATA.mkdir(parents=True, exist_ok=True)

SPECIES_FOLDERS = [
    ('apple dataset', {
        'rust': 'Apple_Rust',
        'scab': 'Apple_Scab',
        'healthy': 'Apple_healthy',
    }),
    ('mango dataset', {
        'Anthracnose': 'Mango_Anthracnose',
        'Bacterial Canker': 'Mango_Bacterial_Canker',
        'Cutting Weevil': 'Mango_Cutting_Weevil',
        'Die Back': 'Mango_Die_Back',
        'Gall Midge': 'Mango_Gall_Midge',
        'Healthy': 'Mango_Healthy',
        'Powdery Mildew': 'Mango_Powdery_Mildew',
        'Sooty Mould': 'Mango_Sooty_Mould',
    }),
    ('Potato Dataset', {
        'Potato_Early_blight': 'Potato_Early_blight',
        'Potato_Late_blight': 'Potato_Late_blight',
        'Potato_healthy': 'Potato_healthy',
    }),
    ('Tomato dataset', {
        'Tomato_Bacterial_spot': 'Tomato_Bacterial_spot',
        'Tomato_Early_blight': 'Tomato_Early_blight',
        'Tomato_Late_blight': 'Tomato_Late_blight',
        'Tomato_Leaf_Mold': 'Tomato_Leaf_Mold',
        'Tomato_Septoria_leaf_spot': 'Tomato_Septoria_leaf_spot',
        'Tomato_Spider_mites_Two_spotted_spider_mite': 'Tomato_Spider_mites_Two_spotted_spider_mite',
        'Tomato_Target_Spot': 'Tomato_Target_Spot',
        'Tomato_Tomato_YellowLeaf_Curl_Virus': 'Tomato_Tomato_YellowLeaf_Curl_Virus',
        'Tomato_Tomato_mosaic_virus': 'Tomato_Tomato_mosaic_virus',
        'Tomato_healthy': 'Tomato_healthy',
    }),
]

copy_jobs = []
tar_bundle = TAR_BUNDLE_PATH if (TAR_BUNDLE_PATH and TAR_BUNDLE_PATH.exists()) else (
    (Path(DRIVE_DATA_DIR).parent / 'datasets_24classes_colab.tar') if DRIVE_DATA_DIR else None
)
if tar_bundle and tar_bundle.exists():
    import tarfile
    print(f"Fast-extracting 24-class bundle ({tar_bundle.name}) to {LOCAL_DATA}...")
    with tarfile.open(tar_bundle, 'r') as tf:
        tf.extractall(LOCAL_DATA)
else:
    for species_dir, class_map in SPECIES_FOLDERS:
        species_path = Path(DRIVE_DATA_DIR) / species_dir
        for sub_name, label_name in class_map.items():
            src_sub = species_path / sub_name
            if not src_sub.exists():
                raise FileNotFoundError(f"Missing expected subfolder: {src_sub}")
            dest_dir = LOCAL_DATA / label_name
            dest_dir.mkdir(parents=True, exist_ok=True)
            for f in src_sub.iterdir():
                if f.is_file() and f.suffix.lower() in ('.jpg', '.jpeg', '.png'):
                    copy_jobs.append((f, dest_dir / f.name))
    print(f"Staging {len(copy_jobs):,} images from {DRIVE_DATA_DIR} to {LOCAL_DATA} with 16 threads...")
    def _copy_pair(pair):
        shutil.copy2(pair[0], pair[1])
    with ThreadPoolExecutor(max_workers=16) as ex:
        list(ex.map(_copy_pair, copy_jobs))

print("\n--- Dataset Summary (Single Source of Truth: MobileV4/datasets) ---")
valid_count = 0
total_imgs = 0
for p in sorted(LOCAL_DATA.iterdir()):
    if p.is_dir():
        imgs = [f for f in p.iterdir() if f.is_file() and f.suffix.lower() in ('.jpg', '.jpeg', '.png')]
        valid_count += 1
        total_imgs += len(imgs)
        print(f"  [{valid_count:02d}] {p.name:45}: {len(imgs):5d} images")

print(f"\n✅ Successfully staged {valid_count} classes ({total_imgs:,} images) strictly from MobileV4/datasets!")


In [ ]:
# 5. Stratified 80/20 Train/Val Split, Screen-Resilient Augmentations & MobileNetV4 Initialization
import random
from collections import defaultdict
import timm
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

# Includes mild color/contrast/sharpness jitter so webcam captures of phone screens classify cleanly
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.72, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.25, hue=0.04),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

base_ds = datasets.ImageFolder(str(LOCAL_DATA))
train_full_ds = datasets.ImageFolder(str(LOCAL_DATA), transform=train_tf)
val_full_ds = datasets.ImageFolder(str(LOCAL_DATA), transform=val_tf)

classes = base_ds.classes
num_classes = len(classes)
assert num_classes == 24, f"Expected 24 classes, found {num_classes}"

# Stratified 80/20 split per class (seed=42)
rng = random.Random(42)
by_class = defaultdict(list)
for idx, (_, label) in enumerate(base_ds.samples):
    by_class[label].append(idx)

train_indices, val_indices = [], []
for label, idxs in sorted(by_class.items()):
    rng.shuffle(idxs)
    n_val = max(1, int(round(0.20 * len(idxs))))
    val_indices.extend(idxs[:n_val])
    train_indices.extend(idxs[n_val:])

train_ds = Subset(train_full_ds, train_indices)
val_ds = Subset(val_full_ds, val_indices)

print(f"✅ Total Images: {len(base_ds):,} | Train (80%): {len(train_ds):,} | Val (20%): {len(val_ds):,}")

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = timm.create_model('mobilenetv4_conv_small', pretrained=True, num_classes=num_classes)
model = model.to(device)
print(f"✅ MobileNetV4-Conv-Small initialized with {num_classes} classes on {device}")


In [ ]:
# 6. Train for 15 Epochs with Mixed Precision (AMP) & Cosine Annealing Scheduler
import torch.nn as nn
import time

criterion = nn.CrossEntropyLoss(label_smoothing=0.06)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
EPOCHS = 15
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
use_amp = torch.cuda.is_available()
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

best_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    train_loss, train_corr = 0.0, 0
    for imgs, targets in train_loader:
        imgs, targets = imgs.to(device), targets.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=use_amp):
            preds = model(imgs)
            loss = criterion(preds, targets)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item() * imgs.size(0)
        train_corr += (preds.argmax(1) == targets).sum().item()

    scheduler.step()
    train_acc = 100.0 * train_corr / len(train_ds)

    model.eval()
    val_loss, val_corr = 0.0, 0
    with torch.no_grad():
        for imgs, targets in val_loader:
            imgs, targets = imgs.to(device), targets.to(device)
            with torch.amp.autocast('cuda', enabled=use_amp):
                preds = model(imgs)
                loss = criterion(preds, targets)
            val_loss += loss.item() * imgs.size(0)
            val_corr += (preds.argmax(1) == targets).sum().item()

    val_acc = 100.0 * val_corr / len(val_ds)
    marker = ""
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), str(WORK_ROOT / 'best_mobilenetv4.pth'))
        marker = " ⭐ Best"
    print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] ({time.time()-t0:.1f}s) | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%{marker}")

print(f"\n🎉 15-Epoch Training Complete! Peak Validation Accuracy: {best_acc:.2f}%")


In [ ]:
# 7. Per-Class Accuracy Report, ONNX Export & Save/Download
import onnx

model.load_state_dict(torch.load(str(WORK_ROOT / 'best_mobilenetv4.pth'), map_location=device))
model.eval()

cls_corr = [0] * num_classes
cls_tot = [0] * num_classes
with torch.no_grad():
    for imgs, targets in val_loader:
        imgs, targets = imgs.to(device), targets.to(device)
        preds = model(imgs).argmax(1)
        for t, p in zip(targets.tolist(), preds.tolist()):
            cls_tot[t] += 1
            if t == p:
                cls_corr[t] += 1

print("--- Per-Class Holdout Validation Accuracy (24 Classes) ---")
for i, c in enumerate(classes):
    acc_i = 100.0 * cls_corr[i] / max(1, cls_tot[i])
    print(f"  [{i:02d}] {c:45}: {acc_i:6.2f}% ({cls_corr[i]}/{cls_tot[i]})")
macro_acc = sum(100.0 * cls_corr[i] / max(1, cls_tot[i]) for i in range(num_classes)) / num_classes
print(f"\nOverall Validation Accuracy : {best_acc:.2f}%")
print(f"Macro-Average Accuracy      : {macro_acc:.2f}%\n")

# Export single-file ONNX model (with all weights embedded inside agrilens.onnx)
model.to('cpu')
dummy = torch.randn(1, 3, 224, 224, dtype=torch.float32)
raw_onnx = str(WORK_ROOT / 'agrilens_raw.onnx')
final_onnx = str(WORK_ROOT / 'agrilens.onnx')

torch.onnx.export(
    model,
    dummy,
    raw_onnx,
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}},
    dynamo=False
)

proto = onnx.load(raw_onnx, load_external_data=True)
onnx.save_model(proto, final_onnx, save_as_external_data=False)

labels_file = str(WORK_ROOT / 'labels.txt')
with open(labels_file, 'w', encoding='utf-8') as f:
    for c in classes:
        f.write(f"{c}\n")

# Sync to local/Drive folders if present, otherwise download directly to your computer
synced = False
if DRIVE_DATA_DIR:
    mobilev4_dir = Path(DRIVE_DATA_DIR).parent
    expiscor_root = mobilev4_dir.parent
    for t_dir in [mobilev4_dir, expiscor_root / 'AGRILENS_RaspberryPi']:
        if t_dir.exists():
            shutil.copy2(final_onnx, t_dir / 'agrilens.onnx')
            shutil.copy2(labels_file, t_dir / 'labels.txt')
            print(f"✅ Synced agrilens.onnx & labels.txt -> {t_dir}")
            synced = True

if not synced:
    print("📥 Downloading trained agrilens.onnx and labels.txt to your computer...")
    try:
        from google.colab import files
        files.download(final_onnx)
        files.download(labels_file)
    except Exception as e:
        print(f"Saved at: {final_onnx} and {labels_file} ({e})")
